## Setup

`nltk` needs three data packages the first time you run this: `punkt`/`punkt_tab` (for the tokenizer)
and `stopwords` (for the stopword list). The cell below downloads them if they're missing.

In [ ]:
import re
import html
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download required NLTK data (safe to re-run; skips if already present)
for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    try:
        nltk.data.find(f'tokenizers/{pkg}' if 'punkt' in pkg else f'corpora/{pkg}')
    except LookupError:
        nltk.download(pkg)

pd.set_option('display.max_colwidth', 120)

> **Important — Guys please `keep_default_na=False`** one of our classes is literally named `None`, and by
> default `pandas.read_csv` treats the string `"None"` as a missing value (NaN). A plain
> `pd.read_csv(...)` silently turns the **entire `None` class (~18k rows, our second-largest class)**
> into NaN. Disabling default NA conversion keeps the label intact. We also strip the text column
> separately so genuinely blank posts are still catchable.

In [ ]:
INPUT_PATH = "datasets/processed/mental_health_cleaned.csv"

# keep_default_na=False so the literal "None" label is NOT read as NaN
df = pd.read_csv(INPUT_PATH, keep_default_na=False, na_values=[])

print(f"Loaded {len(df):,} rows")
print("Columns:", list(df.columns))
df.head()

> **Leaving this comment here so everyone is aware (label bug in curation)** the cleaned file currently contains a
> `Personality disorder` label (~895 rows). Siraj's label map keys it as `'Personality Disorder'`
> (capital D) but the raw value is lowercase `'Personality disorder'`, so it never gets remapped to
> `None`. This isn't a preprocessing issue, but I'm flagging it so it's fixed before modeling. The check
> below makes it visible.

In [ ]:
# Visibility check on labels (should be the 6 shared classes)
print(df['label'].value_counts())

In [ ]:
# --- Step 7 resource: English stopwords ---
STOPWORDS = set(stopwords.words('english'))

# --- Compiled patterns ---
URL_RE      = re.compile(r'http\S+|www\.\S+')          # step 3
HTML_TAG_RE = re.compile(r'<[^>]+>')                      # step 4
REDDIT_RE   = re.compile(r'\[removed\]|\[deleted\]', re.IGNORECASE)  # step 4
PUNCT_RE    = re.compile(r"[^a-z0-9'\s]")                # step 5: keep letters, digits, apostrophes, spaces

def preprocess_text(text):
    text = str(text)

    # 2. lowercase
    text = text.lower()

    # 3. remove URLs
    text = URL_RE.sub(' ', text)

    # 4. remove HTML tags + decode HTML entities (&amp;, &#x200b;, ...) + Reddit markup
    text = html.unescape(text)
    text = HTML_TAG_RE.sub(' ', text)
    text = REDDIT_RE.sub(' ', text)

    # 5. remove punctuation / special chars, but KEEP apostrophes for contractions
    text = text.replace('\u2019', "'").replace('\u02bc', "'")  # normalize curly apostrophes
    text = PUNCT_RE.sub(' ', text)

    # 6. tokenize
    tokens = word_tokenize(text)

    # 7. stopword removal (also drop the clitic/punctuation fragments word_tokenize
    #    produces from contractions, e.g. "n't", "'ve", by keeping alphabetic tokens only)
    tokens = [t for t in tokens if t.isalpha() and t not in STOPWORDS]

    return tokens

What the function does to a few representative posts:

In [ ]:
for i in range(3):
    raw = df['text'].iloc[i]
    print("RAW  :", raw[:140])
    print("CLEAN:", " ".join(preprocess_text(raw))[:140])
    print("-" * 80)

In [ ]:
df['tokens'] = df['text'].apply(preprocess_text)
df['clean_text'] = df['tokens'].apply(lambda toks: ' '.join(toks))

# Sanity checks
token_counts = df['tokens'].apply(len)
print(f"Rows processed         : {len(df):,}")
print(f"Avg tokens per post    : {token_counts.mean():.1f}")
print(f"Posts empty after clean: {(token_counts == 0).sum()}  (consider dropping before modeling)")
df[["text", "label", "clean_text"]].head()

In [ ]:
# Save. clean_text is the column the rest of the pipeline builds on.
OUTPUT_PATH = "datasets/processed/mental_health_preprocessed.csv"
df[['text', 'clean_text', 'label', 'source']].to_csv(OUTPUT_PATH, index=False)
print(f"Saved -> {OUTPUT_PATH}")